In [9]:
import pickle
from sentence_transformers import SentenceTransformer, InputExample, losses, models, util, evaluation
from torch.utils.data import DataLoader
import torch
import os

In [11]:
base_path = r"C:\Users\ohs\Desktop\Tasktwo"
output_path = os.path.join(base_path, "cfg")

with open(base_path + r"\kb_definitions.pickle", "rb") as f:
    kb_definitions = pickle.load(f)

with open(base_path + r"\kb_synonyms_labels.pickle", "rb") as f:
    kb_synonyms_labels = pickle.load(f)

In [12]:
first_key = next(iter(kb_definitions))
print(first_key, kb_definitions[first_key])

first_key = next(iter(kb_synonyms_labels))
print(first_key, kb_synonyms_labels[first_key])

http://purl.obolibrary.org/obo/BTO_0000754 Hyaline eosinophilic concentrically-laminated inclusions found in the substantia nigra and locus ceruleus of patients with Parkinsonism and Lewy body dementia.
http://purl.obolibrary.org/obo/BTO_0000754 ['lewy body']


In [13]:
train_data = []

for entity_id, aliases in kb_synonyms_labels.items():
    definition = kb_definitions.get(entity_id, "")
    
    for alias in aliases:
        train_data.append({
            "mention": alias,
            "context": definition,
            "label": entity_id
        })

In [14]:
from sklearn.model_selection import train_test_split

train_data_split, val_data_split = train_test_split(train_data, test_size=0.1)

train_examples = [
    InputExample(texts=[item["mention"], item["context"]], label=1.0)
    for item in train_data_split
]

val_examples = [
    InputExample(texts=[item["mention"], item["context"]], label=1.0)
    for item in val_data_split
]

evaluator = evaluation.EmbeddingSimilarityEvaluator.from_input_examples(
    val_examples,
    name="val-eval"
)

In [15]:
train_loader = DataLoader(train_examples, shuffle=True, batch_size=16)

In [16]:
transformer = models.Transformer("allenai/biomed_roberta_base")


pooling = models.Pooling(
    transformer.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True,
    pooling_mode_cls_token=False,
    pooling_mode_max_tokens=False
)

dense = models.Dense(
    in_features=pooling.get_sentence_embedding_dimension(),
    out_features=256, 
    activation_function=torch.nn.Tanh()
)

model = SentenceTransformer(modules=[transformer, pooling])


loss = losses.MultipleNegativesRankingLoss(model)

In [17]:
model.fit(
    train_objectives=[(train_loader, loss)],
    epochs=8,
    warmup_steps=100,
)

model.save(output_path)

Step,Training Loss
500,1.327000
1000,0.953300
1500,0.880700
2000,0.793000
2500,0.791000
3000,0.762600
3500,0.731200
4000,0.718900
4500,0.706000
5000,0.718300
